# Lesson 20: Data Science in the Wild — Digital Humanities & Open Earth Data

In this notebook, you will:
1. Analyze Emily Dickinson's poetry sentiment with **Google AI Studio Free Tier** (`google-genai`).
2. Query public satellite imagery catalogs using the open **STAC API** (`pystac-client`).

In [ ]:
!pip install -q pystac-client google-genai pandas
from pystac_client import Client

# Open Earth Observation STAC query (no account required)
try:
    catalog = Client.open("https://earth-search.aws.element84.com/v1")
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=[-122.5, 37.7, -122.3, 37.85],
        datetime="2026-10-01/2026-10-31",
        query={"eo:cloud_cover": {"lt": 15}},
        max_items=5
    )
    items = list(search.items())
    print(f"Found {len(items)} low-cloud Sentinel-2 scenes over San Francisco:")
    for item in items:
        print(" -", item.id, "cloud_cover=", item.properties.get("eo:cloud_cover"))
except Exception as e:
    print(f"STAC network query skipped: {e}")

In [ ]:
# Digital Humanities: Emily Dickinson poem tone analysis with Gemini API
poem = "'Hope' is the thing with feathers - That perches in the soul - And sings the tune without the words - And never stops - at all -"
try:
    from google.colab import userdata
    from google import genai
    api_key = userdata.get("GEMINI_API_KEY")
    if api_key:
        client = genai.Client(api_key=api_key)
        res = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=f"Classify the emotional tone and likely season metaphor of this Emily Dickinson stanza in 2 sentences: {poem}"
        )
        print(res.text)
    else:
        print("Set GEMINI_API_KEY in Colab Secrets to run live Gemini literary sentiment analysis.")
except Exception as e:
    print(f"Gemini analysis skipped: {e}")